# AF forecasting: paired prediction audit

**Snapshot: 15 October 2026.** Based on the two supplied validation CSVs. No training, signal extraction, or outer-test evaluation was performed.

## Decision

The anchored checkpoint makes a small ranking improvement on this validation cohort. At exactly matched specificity it gains **one positive window and one event**. The earlier seven-window loss at the baseline's numerical cutoff mostly reflects a score shift. The particularly low legacy event sensitivity, **1/42**, comes from a different threshold-selection population and is exactly reproducible.

These findings justify keeping the anchored checkpoint as a candidate. They do not demonstrate a substantial improvement in forecasting or an independently validated gain.

## 1. Cohort integrity

| Check | Result |
|---|---:|
| Saved baseline rows | 1,664 |
| Saved anchored rows | 2,669 |
| Shared core windows | 1,574 |
| Core positive / negative windows | 185 / 1,389 |
| Core patients | 35: 14 legacy, 21 IRIDIA |
| Positive events | 84: 42 legacy, 42 IRIDIA |
| Extra baseline rows | 90 legacy negatives |
| Extra anchored rows | 1,095 SHDB windows |

`window_id` is unique in each file. Every paired label, subject, source, event ID and lead bin matches, including missing values. Every positive has an event ID. All 1,574 `manifest_index` values differ: this field is an index within a particular manifest, and must not be used to join these files. The CSVs alone do not establish why the 90 baseline negatives were excluded; that requires the eligibility manifest.

## 2. Fair operating-point comparison

Threshold rule: predict positive when `probability >= threshold`; maximize recall subject to specificity >=90%, then maximize F1, then specificity, then choose the smallest threshold. This reproduces the archived evaluator. Thresholds and counts below were independently checked against its implementation.

| Core metric | Baseline | Anchored |
|---|---:|---:|
| Threshold | 0.33056640625 | 0.319091796875 |
| True-positive windows | 56/185 | 57/185 |
| False-positive windows | 138/1,389 | 138/1,389 |
| Recall | 30.2703% | 30.8108% |
| Specificity | 90.0648% | 90.0648% |
| Average precision | 0.23271648 | 0.23563923 |
| AUROC | 0.66234896 | 0.66717841 |
| Standardized partial AUROC, FPR <=0.10 | 0.56828853 | 0.56822504 |
| Positive events detected | 35/84 | 36/84 |

AP is sklearn average precision, not trapezoidal PR area. These baseline figures use the uploaded saved scores; very small differences from a later baseline inference run are expected from the earlier observed numerical differences. The partial-AUROC result shows that the improvement does not extend uniformly across the entire low-FPR region.

At matched core specificity, every IRIDIA binary prediction is identical. Legacy gains one positive and swaps one false-positive window for another. No previously detected positive is lost. The added positive is:

- Patient: `afdb:04126`
- Event: `afdb:04126:onset_8273014`
- Window: `legacy:afdb:04126:7920000`
- Saved lead bin: `12_16`
- Baseline score: 0.325927734375
- Anchored score: 0.31982421875

Its anchored score decreased, but it clears the anchored model's lower, equally specific threshold. This is a threshold-relative improvement, not an increase in that patient's absolute score.

## 3. Why the old numerical cutoff looked worse

At `p > 0.329345703125`, the original results reproduce exactly: baseline TP56/FP138, anchored TP49/FP112. Specificity increases from 90.06% to 91.94%, so this does not compare recall at equal false-positive burden.

Across the 1,574 paired windows:

- Spearman score correlation: **0.999133**.
- **99.62%** of scores decreased.
- **All 185 positive scores decreased**.
- Mean decrease: 0.00899 for positives and 0.00944 for negatives.

The observed output change is predominantly a score shift with small ranking changes. The files cannot identify whether the cause is the anchor, optimizer, sampling, class mixture, or another training detail. A near-identical ranking also leaves little observed complementarity for blending these two checkpoints at the audited operating point.

## 4. Event sensitivity discrepancy resolved

For the **same anchored checkpoint**, only the threshold-selection population changes:

| Threshold population / rule | Threshold | Legacy events | IRIDIA events | SHDB events |
|---|---:|---:|---:|---:|
| Core: legacy + IRIDIA | 0.31909180 | 11/42 (26.19%) | 25/42 (59.52%) | Not reported here |
| All three sources | 0.36767578 | 1/42 (2.38%) | 18/42 (42.86%) | 16/50 (32.00%) |
| Separate threshold per source | Different per source | 14/42 (33.33%) | 17/42 (40.48%) | 8/50 (16.00%) |

The all-three-source row exactly reproduces the previously reported event triplet: legacy 0.0238095, IRIDIA 0.4285714, SHDB 0.32. This establishes that those numbers are compatible with a global all-source threshold, rather than the core threshold used for the core recall headline. It does not prove the unseen current evaluator's internal implementation; the numerical reproduction is exact.

The global all-source threshold reaches 90.34% specificity overall, but its source-specific specificities are **98.89% legacy, 88.69% IRIDIA, and 81.65% SHDB**. Similarly, the core threshold gives about 94.35% specificity on legacy and 79.40% on IRIDIA. Aggregate specificity therefore hides substantial differences between sources.

Separate source thresholds are a diagnostic, not evidence that source-specific deployment thresholds are valid. Choosing them on this validation set is optimistic, and a new deployment population may not have a meaningful matching dataset label. A single threshold selected for a changed source mixture can materially change event detection even when model weights are unchanged.

Event detection here means at least one *saved positive window* for an eligible event crosses the threshold. This is not continuous-stream event sensitivity, false alarms per hour, or guaranteed advance-warning time. The saved files do not provide sufficient continuous timing coverage to establish those metrics.

## 5. Uncertainty

Exploratory paired bootstrap: 2,000 replicates, seed 20261015, resample patients with replacement **within each source**, using identical resampled patients for both models. Reselect each model's threshold within each replicate. Percentile 95% intervals for anchored minus baseline:

| Difference | Estimate | Exploratory 95% interval |
|---|---:|---:|
| AP | +0.002923 | +0.001512 to +0.004884 |
| AUROC | +0.004829 | +0.002579 to +0.007431 |
| Recall at specificity >=90% | +0.005405 | -0.021368 to +0.023672 |

The small ranking gain is consistent under this resampling procedure. The operating-point recall gain remains uncertain. These intervals condition on already trained and selected checkpoints and a repeatedly used validation cohort; they do not account for model/experiment selection, training randomness, or unseen-domain shift. They are not confirmatory significance tests.

## 6. Next experiment and implementation gates

1. **Make the evaluation contract explicit.** Store threshold value, comparison operator, selection population, target specificity and achieved specificity together. Use the same declared threshold for window and event metrics. Always display source-wise outcomes alongside aggregate results. Core and all-source evaluations can both exist, but must be named separately.
2. **Preserve both checkpoints and their original preprocessing.** Keep this anchored checkpoint as a small improvement candidate, and retain the untouched baseline as the reference. The prediction evidence does not support spending the next run on another nearly identical anchored continuation or blending these two scores.
3. **Test a distinct information representation.** Following the earlier code review, extract ordered RR intervals with actual timestamps and validity masks from the same 10-minute observation. Recompute longer-window HRV directly from those beats; avoid averaging short-window LF/HF ratios or compressing timestamps when rejecting intervals. Start with a compact, regularized RR/HRV-only model to test whether that representation can detect events missed by the ECG fusion model. This is a hypothesis, not an established cause of current errors or a promised gain.
4. **Freeze one primary comparison before training.** Retain the existing patient splits and eligibility rules. Fit transformations on training patients. Use a declared validation operating-point rule and require a useful effect, with patient-level uncertainty, rather than accepting a tiny AP increase alone. Restrict architecture and hyperparameter selection to training/grouped inner validation as available. Any final claim needs confirmation on appropriately held-out data without tuning to it.
5. **Integrate using the current notebook.** The two CSVs are sufficient for this audit. They do not contain ECG signals, R-peaks, training code, the active loader, or the current evaluator. The current `.ipynb` is still needed to write a compatible restart-safe extraction/training patch and confirm the exact three-source training/evaluation implementation.

## Reproducibility

The following cells rerun this audit in Colab after a runtime restart. They require only the uploaded prediction ZIP and standard NumPy, pandas, SciPy and scikit-learn packages. They do not load a model or need a GPU. The static results above describe the supplied files; rerunning on different files does not rewrite this narrative. Computed results are saved as `/content/afib_prediction_audit/audit_results.json`.


In [ ]:
from pathlib import Path
import io, zipfile
from google.colab import files

uploaded = files.upload()  # Choose the ZIP containing the two prediction CSVs.
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_files) != 1:
    raise ValueError('Upload exactly one prediction ZIP.')
ROOT = Path('/content/afib_prediction_audit')
(ROOT / 'input').mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_files[0]])) as archive:
    for name in ['baseline_predictions.csv', 'anchored_predictions.csv']:
        members = [info for info in archive.infolist()
                   if not info.is_dir() and Path(info.filename).name == name
                   and '__MACOSX' not in info.filename]
        if len(members) != 1:
            raise ValueError(f'Expected exactly one {name}; found {len(members)}.')
        (ROOT / 'input' / name).write_bytes(archive.read(members[0]))
print('Prediction files ready:', ROOT / 'input')


In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT=Path('/content/afib_prediction_audit')
old=pd.read_csv(ROOT/'input/baseline_predictions.csv')
new=pd.read_csv(ROOT/'input/anchored_predictions.csv')
for d in (old,new):
 assert d.window_id.is_unique
 assert d.label.isin([0,1]).all() and d.probability.between(0,1).all()
 assert d.loc[d.label.eq(1),'event_id'].notna().all()
p=old.merge(new,on='window_id',suffixes=('_old','_new'),validate='one_to_one')
assert len(p)==1574
assert set(p.window_id)==set(new.loc[new.source_family.isin(['legacy','iridia']),'window_id'])
for c in ['subject_id','source_family','label','event_id','lead_bin']:
 assert p[c+'_old'].fillna('NA').equals(p[c+'_new'].fillna('NA')),c
frame=p[['window_id']].copy()
for c in ['subject_id','source_family','label','event_id','lead_bin']:
 frame[c]=p[c+'_new']
frame['baseline']=p.probability_old
frame['anchored']=p.probability_new

def metrics(y,s,t):
 y=np.asarray(y)==1; h=np.asarray(s)>=t
 tp=int((y&h).sum());fp=int((~y&h).sum());P=int(y.sum());N=int((~y).sum())
 return dict(threshold=float(t),TP=tp,FP=fp,positives=P,negatives=N,recall=tp/P,specificity=1-fp/N,precision=tp/(tp+fp) if tp+fp else 0)

def select(y,s):
 # Vectorized exact equivalent of archived select_threshold: >=; maximize
 # recall, then F1, specificity, then choose the smallest threshold.
 y=np.asarray(y); s=np.asarray(s)
 candidates=np.unique(np.r_[0.,s,1.])
 pos=np.sort(s[y==1]);neg=np.sort(s[y==0])
 tp=len(pos)-np.searchsorted(pos,candidates,side='left')
 fp=len(neg)-np.searchsorted(neg,candidates,side='left')
 recall=tp/len(pos);spec=1-fp/len(neg)
 f1=2*tp/(len(pos)+tp+fp)
 valid=np.flatnonzero(spec>=.9)
 order=np.lexsort((-candidates[valid],spec[valid],f1[valid],recall[valid]))
 return metrics(y,s,candidates[valid[order[-1]]])

def rank(y,s):
 return dict(AP=float(average_precision_score(y,s)),AUROC=float(roc_auc_score(y,s)),standardized_partial_AUC_FPR_0_1=float(roc_auc_score(y,s,max_fpr=.1)))

def event_table(d,score,t):
 e=d[d.label.eq(1)].groupby(['source_family','subject_id','event_id'])[score].max()
 return {source:dict(detected=int((v>=t).sum()),events=len(v),sensitivity=float((v>=t).mean())) for source,v in e.groupby(level=0)}

res={'rows':{'baseline_saved':len(old),'anchored_saved':len(new),'paired':len(frame),'patients':frame.subject_id.nunique()},'core':{},'sources':{}}
for model in ['baseline','anchored']:
 res['core'][model]={**rank(frame.label,frame[model]),**select(frame.label,frame[model])}
 res['core'][model]['events']=event_table(frame,model,res['core'][model]['threshold'])
for source,d in frame.groupby('source_family'):
 res['sources'][source]={}
 for model in ['baseline','anchored']:
  own=select(d.label,d[model]);global_t=res['core'][model]['threshold']
  res['sources'][source][model]={'ranking':rank(d.label,d[model]),'at_core_threshold':metrics(d.label,d[model],global_t),'at_source_threshold':own,'events_at_source_threshold':event_table(d,model,own['threshold'])}
all_t=select(new.label,new.probability)
res['anchored_all_sources']={**all_t,'events':event_table(new,'probability',all_t['threshold'])}
res['anchored_all_sources']['by_source']={source:metrics(d.label,d.probability,all_t['threshold']) for source,d in new.groupby('source_family')}
res['anchored_source_events']={source:event_table(d,'probability',select(d.label,d.probability)['threshold']) for source,d in new.groupby('source_family')}
fixed=.329345703125
res['fixed_baseline_cutoff']={model:metrics(frame.label,frame[model],np.nextafter(fixed,np.inf)) for model in ['baseline','anchored']}
res['fixed_baseline_cutoff']['events']={model:event_table(frame,model,np.nextafter(fixed,np.inf)) for model in ['baseline','anchored']}
res['score_changes']={'spearman':float(spearmanr(frame.baseline,frame.anchored).statistic),'fraction_decreased':float((frame.anchored<frame.baseline).mean())}
for label,d in frame.groupby('label'):
 res['score_changes'][str(label)]={'mean_delta':float((d.anchored-d.baseline).mean()),'median_delta':float((d.anchored-d.baseline).median()),'fraction_decreased':float((d.anchored<d.baseline).mean())}
frame['old_hit']=frame.baseline>=res['core']['baseline']['threshold']
frame['new_hit']=frame.anchored>=res['core']['anchored']['threshold']
frame['change']=np.select([frame.old_hit&~frame.new_hit,~frame.old_hit&frame.new_hit],['lost','gained'],default='same')
res['window_changes']={str(label):d.groupby(['source_family','change']).size().to_dict() for label,d in frame.groupby('label')}
res['window_changes']={k:{' / '.join(kk):vv for kk,vv in v.items()} for k,v in res['window_changes'].items()}
e=frame[frame.label.eq(1)].groupby(['source_family','subject_id','event_id'])[['old_hit','new_hit']].max()
e['change']=np.select([e.old_hit&~e.new_hit,~e.old_hit&e.new_hit],['lost','gained'],default='same')
res['event_changes']={' / '.join(k):v for k,v in e.groupby(['source_family','change']).size().to_dict().items()}
res['changed_events']=e[e.change.ne('same')].reset_index().to_dict(orient='records')
res['changed_positive_windows']=frame[frame.label.eq(1)&frame.change.ne('same')][['window_id','subject_id','source_family','event_id','lead_bin','baseline','anchored','change']].to_dict(orient='records')
res['lead_bins']=frame[frame.label.eq(1)].groupby(['source_family','lead_bin']).agg(positive_windows=('label','size'),baseline_detected=('old_hit','sum'),anchored_detected=('new_hit','sum')).reset_index().to_dict(orient='records')
# Paired patient bootstrap, sampling patients with replacement within source.
# Refit each model's validation threshold within each replicate.
rng=np.random.default_rng(20261015)
clusters=[list(d.groupby('subject_id').indices.values()) for _,d in frame.groupby('source_family')]
# indices above local to each source: build global positions explicitly.
clusters=[]
for source,d in frame.groupby('source_family'):
 clusters.append([g.index.to_numpy() for _,g in d.groupby('subject_id')])
y=frame.label.to_numpy();a=frame.baseline.to_numpy();b=frame.anchored.to_numpy()
boot=[]
for _ in range(2000):
 idx=np.concatenate([np.concatenate([cs[i] for i in rng.integers(0,len(cs),len(cs))]) for cs in clusters])
 yy=y[idx];aa=a[idx];bb=b[idx]
 if len(np.unique(yy))<2: continue
 boot.append([average_precision_score(yy,bb)-average_precision_score(yy,aa),roc_auc_score(yy,bb)-roc_auc_score(yy,aa),select(yy,bb)['recall']-select(yy,aa)['recall']])
boot=np.asarray(boot)
res['paired_patient_bootstrap']={'replicates':len(boot),'seed':20261015,'method':'patients resampled within source; same patients for both models; thresholds reselected in each replicate; percentile interval; exploratory post-selection','delta_95_percent_intervals':{name:np.quantile(boot[:,i],[.025,.975]).tolist() for i,name in enumerate(['AP','AUROC','recall_at_specificity_floor'])}}
res['file_sha256']={f.name:hashlib.sha256(f.read_bytes()).hexdigest() for f in (ROOT/'input').glob('*.csv')}
(ROOT/'audit_results.json').write_text(json.dumps(res,indent=2,default=lambda v:v.item() if hasattr(v,'item') else str(v)))
display(pd.DataFrame(res['core']).T.drop(columns='events'))
print('Core events:', {k: v['events'] for k,v in res['core'].items()})
print('All-source anchored events:', res['anchored_all_sources']['events'])
print('Changed events:', res['changed_events'])
print('Bootstrap:', res['paired_patient_bootstrap'])
print('Full results saved to:', ROOT / 'audit_results.json')


## Download the computed results

Run this cell after the audit finishes.

In [ ]:
files.download(str(ROOT / 'audit_results.json'))